# Netflix Content Analysis: Data Cleaning
**Goal:** turn the raw Kaggle Netflix catalogue into a clean dataset ready for analysis.

**Input:** `data/raw/netflix_titles.csv` ([Kaggle: Netflix Movies and TV Shows](https://www.kaggle.com/datasets/shivamb/netflix-shows), catalogue as of 2021)  
**Output:** `data/processed/netflix_clean.csv`, `netflix_genres.csv`, `netflix_countries.csv`

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/netflix_titles.csv')
print(df.shape)
df.head()

(8807, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


## 1. First checks

In [3]:
df.isnull().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

In [4]:
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate show_id:', df['show_id'].duplicated().sum())
print('Same title, type and release year:', df.duplicated(['title', 'type', 'release_year']).sum())

Duplicate rows: 0
Duplicate show_id: 0
Same title, type and release year: 0


**Findings**
- 8,807 titles and 12 columns, with no duplicates.
- Missing values: `director` (2,634, 30%), `cast` (825), `country` (831), `date_added` (10), `rating` (4), `duration` (3).
- `date_added` is text, `duration` mixes minutes and seasons, and `listed_in` / `country` hold comma-separated lists.

## 2. Fix shifted values in `rating`
Three stand-up specials have their duration typed into the `rating` column, leaving `duration` empty.

In [5]:
df['rating'].value_counts().tail(6)

rating
TV-Y7-FV    6
NC-17       3
UR          3
74 min      1
84 min      1
66 min      1
Name: count, dtype: int64

In [6]:
shifted = df['rating'].str.contains('min', na=False)
df[shifted][['title', 'rating', 'duration']]

,title,rating,duration
5541,Louis C.K. 2017,74 min,NaN
5794,Louis C.K.: Hilarious,84 min,NaN
5813,Louis C.K.: Live at the Comedy Store,66 min,NaN


In [7]:
df.loc[shifted, 'duration'] = df.loc[shifted, 'rating']
df.loc[shifted, 'rating'] = np.nan
df.loc[shifted, ['title', 'rating', 'duration']]

,title,rating,duration
5541,Louis C.K. 2017,NaN,74 min
5794,Louis C.K.: Hilarious,NaN,84 min
5813,Louis C.K.: Live at the Comedy Store,NaN,66 min


## 3. Handle missing values
`director`, `cast` and `country` are missing for many titles, especially TV shows. Dropping those rows would lose a third of the catalogue, so they are filled with **"Unknown"** and flagged. Missing ratings become **"Not Rated"**.

In [8]:
for col in ['director', 'cast', 'country']:
    df[f'{col}_missing'] = df[col].isnull().astype(int)
    df[col] = df[col].fillna('Unknown')

df['rating'] = df['rating'].fillna('Not Rated')

print(df.groupby('type')[['director_missing', 'cast_missing', 'country_missing']].mean().round(2))

         director_missing  cast_missing  country_missing
type                                                    
Movie                0.03          0.08             0.07
TV Show              0.91          0.13             0.15


**Finding:** 91% of TV shows have no director listed, against 3% of movies. TV shows are usually credited to creators rather than directors, so a missing director is expected, not an error.

## 4. Convert `date_added` to a date
Some values have a leading space, so they are stripped first.

In [9]:
print('Values with a leading space:', df['date_added'].str.startswith(' ').sum())
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), format='%B %d, %Y', errors='coerce')
print('Missing dates:', df['date_added'].isnull().sum())

df['year_added'] = df['date_added'].dt.year.astype('Int64')
df['month_added'] = df['date_added'].dt.month_name()
df[['date_added', 'year_added', 'month_added']].head()

Values with a leading space: 88
Missing dates: 10


,date_added,year_added,month_added
0,2021-09-25,2021,September
1,2021-09-24,2021,September
2,2021-09-24,2021,September
3,2021-09-24,2021,September
4,2021-09-24,2021,September


The 10 titles without a date are kept; they are only excluded from time-based charts.

## 5. Split `duration` into a number and a unit
Movies are measured in minutes and TV shows in seasons, so they become separate columns.

In [10]:
parts = df['duration'].str.split(' ', expand=True)
df['duration_value'] = pd.to_numeric(parts[0])
df['movie_minutes'] = df['duration_value'].where(df['type'] == 'Movie')
df['tv_seasons'] = df['duration_value'].where(df['type'] == 'TV Show')
df[['type', 'duration', 'movie_minutes', 'tv_seasons']].sample(5, random_state=1)

,type,duration,movie_minutes,tv_seasons
2894,TV Show,1 Season,NaN,1.0
2161,Movie,72 min,72.0,NaN
6363,Movie,74 min,74.0,NaN
2575,TV Show,1 Season,NaN,1.0
2698,TV Show,2 Seasons,NaN,2.0


## 6. Group ratings by audience
Netflix mixes TV ratings (TV-MA, TV-14…) and film ratings (R, PG-13…). Grouping them by audience makes them comparable.

In [11]:
audience = {
    'TV-Y': 'Kids', 'TV-Y7': 'Kids', 'TV-Y7-FV': 'Kids', 'TV-G': 'Kids', 'G': 'Kids',
    'PG': 'Older Kids', 'TV-PG': 'Older Kids',
    'PG-13': 'Teens', 'TV-14': 'Teens',
    'R': 'Adults', 'TV-MA': 'Adults', 'NC-17': 'Adults',
    'NR': 'Not Rated', 'UR': 'Not Rated', 'Not Rated': 'Not Rated'}
df['audience'] = df['rating'].map(audience)
print(df['audience'].isnull().sum(), 'unmapped ratings')
df['audience'].value_counts()

0 unmapped ratings


audience
Adults        4009
Teens         2650
Older Kids    1150
Kids           908
Not Rated       90
Name: count, dtype: int64

## 7. New columns for analysis

In [12]:
df['primary_country'] = df['country'].str.split(',').str[0].str.strip()
df['primary_genre'] = df['listed_in'].str.split(',').str[0].str.strip()
df['genre_count'] = df['listed_in'].str.split(',').str.len()
df['years_to_netflix'] = (df['year_added'] - df['release_year']).astype('Int64')
df[['title', 'primary_country', 'primary_genre', 'release_year', 'year_added', 'years_to_netflix']].head()

,title,primary_country,primary_genre,release_year,year_added,years_to_netflix
0,Dick Johnson Is Dead,United States,Documentaries,2020,2021,1
1,Blood & Water,South Africa,International TV Shows,2021,2021,0
2,Ganglands,Unknown,Crime TV Shows,2021,2021,0
3,Jailbirds New Orleans,Unknown,Docuseries,2021,2021,0
4,Kota Factory,India,International TV Shows,2021,2021,0


In [13]:
# A few titles show a negative gap (added before their release year): data entry quirks
print('Negative gaps:', (df['years_to_netflix'] < 0).sum())
df.loc[df['years_to_netflix'] < 0, 'years_to_netflix'] = pd.NA

Negative gaps: 14


## 8. One-row-per-value tables for genres and countries
A title can belong to several genres and countries (e.g. "Dramas, International Movies"). To count them correctly, each is split into its own table.

In [14]:
genres = df[['show_id', 'type', 'listed_in']].copy()
genres['genre'] = genres['listed_in'].str.split(',')
genres = genres.explode('genre')
genres['genre'] = genres['genre'].str.strip()
genres = genres[['show_id', 'type', 'genre']]

countries = df[['show_id', 'type', 'country']].copy()
countries['country_name'] = countries['country'].str.split(',')
countries = countries.explode('country_name')
countries['country_name'] = countries['country_name'].str.strip()
countries = countries[countries['country_name'] != ''][['show_id', 'type', 'country_name']]

print('Genre rows:', len(genres), '| unique genres:', genres['genre'].nunique())
print('Country rows:', len(countries), '| unique countries:', countries['country_name'].nunique())

Genre rows: 19323 | unique genres: 42
Country rows: 10843 | unique countries: 123


## 9. Validate and save

In [15]:
assert df['show_id'].is_unique
assert df['type'].isin(['Movie', 'TV Show']).all()
assert df['duration_value'].notnull().all()
assert df['audience'].notnull().all()
print('All checks passed')

df.to_csv('../data/processed/netflix_clean.csv', index=False)
genres.to_csv('../data/processed/netflix_genres.csv', index=False)
countries.to_csv('../data/processed/netflix_countries.csv', index=False)
print(df.shape, genres.shape, countries.shape)

All checks passed


(8807, 25) (19323, 3) (10843, 3)


## Cleaning summary
| Step | Result |
|---|---|
| Raw data | 8,807 titles, 12 columns, no duplicates |
| Shifted values | 3 durations moved from `rating` back to `duration` |
| Missing director / cast / country | Filled with "Unknown" and flagged (not dropped) |
| Missing rating | 7 set to "Not Rated" |
| `date_added` | Converted to a date; 88 values had a leading space; 10 missing kept |
| `duration` | Split into `movie_minutes` and `tv_seasons` |
| New columns | `year_added`, `month_added`, `audience`, `primary_country`, `primary_genre`, `genre_count`, `years_to_netflix` |
| Extra tables | One row per title per genre, and per title per country |